# CAMELS Credit Analyst — QLoRA Fine-tuning
## Google Colab Training Notebook

**Instructions:**
1. Runtime → Change runtime type → GPU (T4 free, or A100 with Colab Pro)
2. Run all cells in order
3. Upload your training data when prompted (Cell 3)
4. Download the trained adapter when complete (Cell 7)

**Files needed from your Mac:**
- `training_data/combined_training_upgraded.jsonl`
- `training_data/combined_eval_upgraded.jsonl`

**Expected time:**
- Free T4 (15GB VRAM): ~45–60 minutes
- Colab Pro A100 (40GB VRAM): ~15–20 minutes

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → GPU')

gpu_name = torch.cuda.get_device_properties(0).name
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU  : {gpu_name}')
print(f'VRAM : {vram_gb:.1f}GB')
print(f'CUDA : {torch.version.cuda}')

# Select model based on VRAM
if vram_gb >= 35:
    MODEL_NAME = 'Qwen/Qwen2.5-14B-Instruct'
    MAX_SEQ    = 3072
    LORA_RANK  = 16
    NUM_LAYERS = 16
    print(f'\nSelected: Qwen2.5-14B (VRAM >= 35GB)')
else:
    MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
    MAX_SEQ    = 2048
    LORA_RANK  = 8
    NUM_LAYERS = 8
    print(f'\nSelected: Qwen2.5-7B (VRAM < 35GB)')

print(f'Max seq length : {MAX_SEQ}')
print(f'LoRA rank      : {LORA_RANK}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# Unsloth provides 2-5x faster training with lower VRAM usage
print('Installing Unsloth and dependencies (~3 minutes)...')

import subprocess
subprocess.run([
    'pip', 'install',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git',
    '--quiet'
], check=True)

subprocess.run([
    'pip', 'install', '--no-deps',
    'trl', 'peft', 'accelerate', 'bitsandbytes', 'datasets',
    '--quiet'
], check=True)

print('Installation complete.')

In [ ]:
# ── Cell 3: Upload training data ──────────────────────────────────────────────
# Option A: Upload directly (simpler)
from google.colab import files
import os

os.makedirs('training_data', exist_ok=True)

print('Upload your two JSONL files from your Mac:')
print('  1. combined_training_upgraded.jsonl')
print('  2. combined_eval_upgraded.jsonl')
print()
print('Select both files when the upload dialog opens...')

uploaded = files.upload()

for filename, content in uploaded.items():
    dest = f'training_data/{filename}'
    with open(dest, 'wb') as f:
        f.write(content)
    lines = content.decode('utf-8').count('\n')
    print(f'Saved: {dest} ({lines} lines)')

# Verify files are present
train_path = 'training_data/combined_training_upgraded.jsonl'
eval_path  = 'training_data/combined_eval_upgraded.jsonl'

if not os.path.exists(train_path):
    # Try without _upgraded suffix
    train_path = 'training_data/combined_training.jsonl'
    eval_path  = 'training_data/combined_eval.jsonl'

train_lines = sum(1 for _ in open(train_path))
eval_lines  = sum(1 for _ in open(eval_path))
print(f'\nTraining pairs : {train_lines}')
print(f'Eval pairs     : {eval_lines}')
print('Ready to train.')

In [ ]:
# ── Cell 4: Load model ────────────────────────────────────────────────────────
from unsloth import FastLanguageModel

print(f'Loading {MODEL_NAME}...')
print('(Downloads ~15GB on first run — takes ~10 minutes)')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length = MAX_SEQ,
    dtype          = None,       # auto-detect bf16/fp16
    load_in_4bit   = True,       # 4-bit quantisation — essential for T4
)

print(f'Model loaded. Applying LoRA adapters...')

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha     = LORA_RANK * 2,
    lora_dropout   = 0.05,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
    max_seq_length = MAX_SEQ,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({100*trainable/total:.3f}%)')
print('Model ready.')

In [ ]:
# ── Cell 5: Prepare dataset ───────────────────────────────────────────────────
from datasets import load_dataset

def format_messages(example):
    """Convert messages list to Qwen2.5 chat template format."""
    messages = example.get('messages', [])
    text = ''
    for msg in messages:
        role    = msg.get('role', '')
        content = msg.get('content', '')
        if role == 'system':
            text += f'<|im_start|>system\n{content}<|im_end|>\n'
        elif role == 'user':
            text += f'<|im_start|>user\n{content}<|im_end|>\n'
        elif role == 'assistant':
            text += f'<|im_start|>assistant\n{content}<|im_end|>\n'
    return {'text': text}

dataset = load_dataset(
    'json',
    data_files={
        'train': train_path,
        'test':  eval_path,
    }
)

dataset = dataset.map(format_messages, remove_columns=dataset['train'].column_names)

print(f'Train: {len(dataset["train"])} examples')
print(f'Test:  {len(dataset["test"])} examples')
print(f'\nSample (first 300 chars):')
print(dataset['train'][0]['text'][:300])

In [ ]:
# ── Cell 6: Train ─────────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments

# Adjust batch size based on VRAM
batch_size = 2 if vram_gb >= 20 else 1
grad_accum = 8 if vram_gb >= 20 else 16

print(f'Batch size      : {batch_size}')
print(f'Grad accumulation: {grad_accum}')
print(f'Effective batch : {batch_size * grad_accum}')
print(f'Starting training...')
print()

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = dataset['train'],
    eval_dataset      = dataset['test'],
    dataset_text_field = 'text',
    max_seq_length    = MAX_SEQ,
    dataset_num_proc  = 2,
    args = TrainingArguments(
        per_device_train_batch_size  = batch_size,
        gradient_accumulation_steps  = grad_accum,
        warmup_steps                 = 20,
        num_train_epochs             = 3,
        learning_rate                = 2e-5,
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        logging_steps                = 10,
        eval_strategy                = 'steps',
        eval_steps                   = 50,
        save_steps                   = 100,
        save_total_limit             = 3,
        output_dir                   = 'models/camels-adapter',
        load_best_model_at_end       = True,
        metric_for_best_model        = 'eval_loss',
        report_to                    = 'none',
        seed                         = 42,
    ),
)

stats = trainer.train()

print(f'\nTraining complete!')
print(f'Steps    : {stats.global_step}')
print(f'Train loss: {stats.training_loss:.4f}')

In [ ]:
# ── Cell 7: Save adapter and download ────────────────────────────────────────
import os, zipfile
from google.colab import files

# Save LoRA adapter
adapter_path = 'camels-analyst-lora'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'Adapter saved: {adapter_path}/')

# Also save merged model (optional — large, ~15GB)
# model.save_pretrained_merged('camels-analyst-merged', tokenizer, save_method='merged_16bit')

# Save GGUF for direct Ollama use (recommended)
print('Saving GGUF (Q4_K_M) for Ollama...')
model.save_pretrained_gguf(
    'camels-analyst-gguf',
    tokenizer,
    quantization_method = 'q4_k_m',
)
print('GGUF saved.')

# Zip everything for download
print('Zipping for download...')
with zipfile.ZipFile('camels_adapter.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [adapter_path, 'camels-analyst-gguf']:
        for root, dirs, filenames in os.walk(folder):
            for filename in filenames:
                filepath = os.path.join(root, filename)
                zf.write(filepath)
                print(f'  Added: {filepath}')

size_mb = os.path.getsize('camels_adapter.zip') / 1_048_576
print(f'\nZip size: {size_mb:.0f}MB')
print('Downloading...')
files.download('camels_adapter.zip')
print('Done! Extract the zip on your Mac.')

In [ ]:
# ── Cell 8: Deploy instructions ──────────────────────────────────────────────
print("""
═══════════════════════════════════════════════════════════
 AFTER DOWNLOADING camels_adapter.zip TO YOUR MAC:
═══════════════════════════════════════════════════════════

 1. Extract the zip:
    unzip camels_adapter.zip -d llm_credit_paper/models/

 2. Update Modelfile to point to the GGUF:
    Edit llm_credit_paper/Modelfile
    Change FROM line to:
    FROM ./models/camels-analyst-gguf/camels-analyst-q4_k_m.gguf

 3. Create Ollama model:
    ollama create camels-analyst-7b -f ./Modelfile

 4. Start server:
    ollama serve

 5. Run analysis:
    python main.py \\
      --pdf financials/lloyds_2025.pdf \\
      --bank 'Lloyds Banking Group'

═══════════════════════════════════════════════════════════
""")